# pipelineA.py - For Standard Models

In [19]:
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
import joblib

## Load data

In [20]:
train = pd.read_csv('dataset/train.csv')
test = pd.read_csv('dataset/test.csv')

y_train = train['demand']
test_index = test['Index']

## Drop the columns we DO NOT want (including geohash for these models)

In [21]:
X_train_raw = train.drop(columns=['demand', 'Index', 'geohash'])
X_test_raw = test.drop(columns=['Index', 'geohash'])

In [22]:
print("--- PIPELINE A: Extracting time features ---")
def extract_time(df):
    df = df.copy()
    df[['hour', 'minute']] = df['timestamp'].str.split(':', expand=True).astype(float)
    return df.drop(columns=['timestamp'])

X_train = extract_time(X_train_raw)
X_test = extract_time(X_test_raw)

--- PIPELINE A: Extracting time features ---


In [23]:
print("--- PIPELINE A: Building and Running Pipeline ---")
numeric_features = ['Temperature', 'NumberofLanes', 'day', 'hour', 'minute']
categorical_features = ['RoadType', 'LargeVehicles', 'Landmarks', 'Weather']

--- PIPELINE A: Building and Running Pipeline ---


In [24]:
num_transformer = SimpleImputer(strategy='median')
cat_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='Missing')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False, drop='first'))
])

In [25]:
pipeline_A = ColumnTransformer(transformers=[
    ('num', num_transformer, numeric_features),
    ('cat', cat_transformer, categorical_features)
])

## Process the data

In [26]:
X_train_encoded = pipeline_A.fit_transform(X_train)
X_test_encoded = pipeline_A.transform(X_test)

## # Convert back to DataFrame

In [27]:
ohe_cols = pipeline_A.named_transformers_['cat'].named_steps['onehot'].get_feature_names_out(categorical_features)
all_columns = numeric_features + list(ohe_cols)

df_train = pd.DataFrame(X_train_encoded, columns=all_columns)
df_train['demand'] = y_train.values

df_test = pd.DataFrame(X_test_encoded, columns=all_columns)
df_test['Index'] = test_index.values

In [28]:
print("--- PIPELINE A: Saving outputs ---")
df_train.to_csv('cleanedA/Train_Standard_Models.csv', index=False)
df_test.to_csv('cleanedA/Test_Standard_Models.csv', index=False)
joblib.dump(pipeline_A, 'Pipeline_A_Preprocessor.pkl')

--- PIPELINE A: Saving outputs ---


['Pipeline_A_Preprocessor.pkl']

In [29]:
print("Pipeline A Complete! CSVs ready for XGBoost, LightGBM, and Random Forest.")

Pipeline A Complete! CSVs ready for XGBoost, LightGBM, and Random Forest.
